In [ ]:
mport os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [ ]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=30)
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    
    return mfcc_mean

In [ ]:
base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

X = []
y = []

genres = os.listdir(base_path)

for genre in genres:
    genre_path = os.path.join(base_path, genre)
    songs = os.listdir(genre_path)
    
    for song in tqdm(songs[:50]):  # initially 50 (later full)
        file_path = os.path.join(genre_path, song, "vocals.wav")
        
        try:
            features = extract_features(file_path)
            X.append(features)
            y.append(genre)
        except:
            continue

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
import wandb

wandb.init(
    project="23f2000441-t12026",
    name="Milestone-2-ML"
)


import warnings
warnings.filterwarnings("ignore")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 1. Scaling (IMPORTANT)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 2. Model
model_lr = LogisticRegression(max_iter=300, solver='liblinear')

# 3. Train
model_lr.fit(X_train_scaled, y_train)

# 4. Predict
y_pred = model_lr.predict(X_val_scaled)

# 5. F1 Score
f1 = f1_score(y_val, y_pred, average="macro")

print("✅ Logistic Regression Macro F1:", f1)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

model_cat = CatBoostClassifier(verbose=0)

model_cat.fit(X_train, y_train_enc)

pred_cat = model_cat.predict(X_val)

f1_cat = f1_score(y_val_enc, pred_cat, average="macro")

print("CatBoost F1:", f1_cat)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score

model_nb = GaussianNB()
model_nb.fit(X_train, y_train)

pred_nb = model_nb.predict(X_val)

f1_nb = f1_score(y_val, pred_nb, average="macro")

print("Naive Bayes F1:", f1_nb)

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

model_xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")

model_xgb.fit(X_train, y_train_enc)

pred_xgb = model_xgb.predict(X_val)

f1_xgb = f1_score(y_val_enc, pred_xgb, average="macro")

print("XGBoost F1:", f1_xgb)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

model_lgb = LGBMClassifier()

model_lgb.fit(X_train, y_train_enc)

pred_lgb = model_lgb.predict(X_val)

f1_lgb = f1_score(y_val_enc, pred_lgb, average="macro")

print("LightGBM F1:", f1_lgb)

In [ ]:
results = {}

# Logistic (agar already run kiya hai toh skip, warna run karo)
try:
    results["Logistic"] = f1
except:
    print("⚠ Logistic not found")

# Naive Bayes
try:
    results["NaiveBayes"] = f1_nb
except:
    print("⚠ NaiveBayes not found")

# XGBoost
try:
    results["XGBoost"] = f1_xgb
except:
    print("⚠ XGBoost not found")

# LightGBM
try:
    results["LightGBM"] = f1_lgb
except:
    print("⚠ LightGBM not found")

# CatBoost
try:
    results["CatBoost"] = f1_cat
except:
    print("⚠ CatBoost not found")

# Print results
print("\n🔥 FINAL MODEL COMPARISON 🔥\n")

for model, score in results.items():
    print(f"{model}: {score:.4f}")

In [ ]:
import pandas as pd

test_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
test_df = pd.read_csv(test_path)

print(test_df.head())

In [ ]:
import os

mashup_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

predictions = []

for file_id in test_df["id"]:
    
    # 🔥 FIX: correct filename format
    file_name = f"song{int(file_id):04d}.wav"
    
    file_path = os.path.join(mashup_path, file_name)
    
    try:
        features = extract_features(file_path)
        features = features.reshape(1, -1)
        
        pred = model_cat.predict(features)
        
        genre = le.inverse_transform(pred)[0]
        
    except Exception as e:
        print("Error:", e)
        genre = "rock"
    
    predictions.append(genre)

test_df["genre"] = predictions

In [ ]:
submission = test_df[["id", "genre"]]

submission.to_csv("submission.csv", index=False)

print("✅ submission.csv created!")